# ZeroGPT Kaggle runner

This notebook reproduces the repository's corpus preparation, character bigram baseline, and tiny GPT run. Make the repository available as a Kaggle dataset or upload it into the working directory before running the cells.

The NumPy implementation runs on the Kaggle CPU; the T4 does not accelerate these operations.

In [ ]:
from pathlib import Path
import os
import shutil

candidates = [Path.cwd(), Path('/kaggle/working/ZeroGPT'), Path('/kaggle/input/zerogpt')]
candidates += [p for base in (Path('/kaggle/working'), Path('/kaggle/input')) if base.exists() for p in base.iterdir()]
source_root = next((p for p in candidates if (p / 'roadmap.md').exists() and (p / 'dataset.txt').exists()), None)
if source_root is None:
    raise FileNotFoundError('Make the ZeroGPT repository available in the Kaggle working directory or as a dataset.')
if str(source_root).startswith('/kaggle/input/'):
    ROOT = Path('/kaggle/working/ZeroGPT')
    if ROOT.exists():
        shutil.rmtree(ROOT)
    shutil.copytree(source_root, ROOT)
else:
    ROOT = source_root
os.chdir(ROOT)
print('Repository root:', ROOT)

In [ ]:
!python -m pip install -q -r requirements.txt

## Prepare the document-ordered split

In [ ]:
!python data/prepare.py

## Train the count-based baseline

In [ ]:
!python scripts/train_bigram.py --output-dir runs/bigram

## Train the tiny GPT

The configuration matches the saved showcase run: seed 1337, 1,200 steps, 2 decoder layers, 4 heads, width 32, and context length 32.

In [ ]:
!python scripts/train_gpt.py --steps 1200 --batch-size 16 --eval-batch-size 64 --eval-interval 100 --sample-tokens 400 --output-dir runs/gpt

In [ ]:
import json
from pathlib import Path

bigram = json.loads(Path('runs/bigram/results.json').read_text())
gpt = json.loads(Path('runs/gpt/results.json').read_text())
print('Bigram validation loss:', bigram['validation_loss_nats'])
print('GPT validation loss:', gpt['validation_loss_nats'])
print('GPT beats bigram:', gpt['beats_bigram'])
print('GPT checkpoint max logit difference:', gpt['checkpoint_max_logit_difference'])
print(Path('runs/gpt/sample.txt').read_text())

In [ ]:
import shutil

artifact = shutil.make_archive('zerogpt_kaggle_runs', 'zip', root_dir='runs')
print('Saved Kaggle artifact:', artifact)